# Honest household-level model

`linear_not_neural.ipynb` reported 96.3% accuracy for Random Forest, but that model was trained
on vehicle-level rows (one row per vehicle, 14,684 of them) rather than one row per household,
and it kept `VEHFUEL`/`VEHTYPE`/`MAKE` in the feature set. `VEHFUEL` codes 4, 5, and 6 mean
hybrid/plug-in-hybrid/electric and predict `HYBRID == 1` with 100% certainty on their own, so the
model was substantially reading the answer off the vehicle's own fuel code rather than learning
from household demographics. Full writeup in `reference/CHALLENGES.md`.

This notebook rebuilds the pipeline properly:

1. Collapse to one row per household **without** the arbitrariness of `.groupby('HOUSEID').first()`
   on vehicle-specific columns (demonstrated below with a real example).
2. Train on demographics/geography alone (matches the model behind `app/streamlit_app.py`, 0.68 ROC-AUC).
3. Add vehicle-fleet features that don't leak `HYBRID`/`VEHFUEL` (fleet age, mileage, variety) and
   see how much real signal that adds.

## Load and merge

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

household_data = pd.read_csv('../data/hhpub.csv')
vehicle_data = pd.read_csv('../data/vehpub.csv')
df = pd.merge(vehicle_data, household_data, on='HOUSEID', how='inner')
duplicates = [c for c in df.columns if c.endswith('_x')]
df = df.drop(columns=duplicates)
df = df.rename(columns={c: c.replace('_y', '') for c in df.columns})
df['HAS_HYBRID'] = df.groupby('HOUSEID')['HYBRID'].transform(lambda x: 1 if (x == 1).any() else 0)
print(f"{len(df)} vehicle rows across {df['HOUSEID'].nunique()} households")

14684 vehicle rows across 7417 households


## Why `.groupby('HOUSEID').first()` is the wrong tool here

A household's vehicle rows agree on every household-level column (income, size, urban/rural...),
those are broadcast copies from the household file, so taking `.first()` of *those* columns is
safe. But vehicle-specific columns (`VEHFUEL`, `VEHTYPE`, `VEHAGE`, `ANNMILES`) genuinely differ
per vehicle within a household, and `.first()` just keeps whichever vehicle happened to be listed
first in the file, arbitrary, and inconsistent depending on which vehicle that happens to be.

In [ ]:
# Every vehicle row for a given household agrees on these columns (max nunique == 1),
# so deduplicating on them is safe, not arbitrary.
household_cols = ['HHFAMINC_IMP', 'HHSIZE', 'HHVEHCNT', 'URBRUR', 'URBANSIZE',
                   'DRVRCNT', 'HOMEOWN', 'LIF_CYC', 'WRKCOUNT', 'MSASIZE', 'CNTTDHH']
nunique_per_house = df.groupby('HOUSEID')[household_cols].nunique()
print("Max distinct values per household per column (1 = safe to dedup):")
print(nunique_per_house.max())

Max distinct values per household per column (1 = safe to dedup):
HHFAMINC_IMP    1
HHSIZE          1
HHVEHCNT        1
URBRUR          1
URBANSIZE       1
DRVRCNT         1
HOMEOWN         1
LIF_CYC         1
WRKCOUNT        1
MSASIZE         1
CNTTDHH         1
dtype: int64


In [ ]:
# Contrast: a household that owns one gas car and one hybrid.
# .first() keeps whichever vehicle is listed first, here that's the gas car,
# and the hybrid vehicle's own row (and its info) just gets discarded.
example = df[df['HOUSEID'] == 9000013167][['HOUSEID', 'VEHID', 'VEHFUEL', 'VEHTYPE', 'ANNMILES', 'VEHAGE', 'HYBRID', 'HAS_HYBRID']]
print(example.to_string(index=False))
print()
print(".first() would keep only:")
print(example.groupby('HOUSEID').first().to_string())

   HOUSEID  VEHID  VEHFUEL  VEHTYPE  ANNMILES  VEHAGE  HYBRID  HAS_HYBRID
9000013167      1        1        4       300       2       2           1
9000013167      2        6        3       320       1       1           1

.first() would keep only:
            VEHID  VEHFUEL  VEHTYPE  ANNMILES  VEHAGE  HYBRID  HAS_HYBRID
HOUSEID                                                                  
9000013167      1        1        4       300       2       2           1


The household is correctly labeled `HAS_HYBRID = 1`, but `.first()` keeps the gas car's `VEHFUEL`,
`VEHTYPE`, `ANNMILES`, and `VEHAGE`, the hybrid's own attributes are gone. 64.6% of households
(4,794 of 7,417) own more than one vehicle, so this isn't an edge case.

**The fix used below:** dedup on the household-level columns only (safe, not arbitrary), and
handle vehicle-level information separately as proper aggregates instead of an arbitrary pick.

## Step 1: safe household-level dedup

In [ ]:
household_df = (
    df[['HOUSEID', 'HAS_HYBRID'] + household_cols]
    .drop_duplicates(subset='HOUSEID')
    .reset_index(drop=True)
)
household_df['URBRUR_BIN'] = household_df['URBRUR'].apply(lambda x: 1 if x == 1 else 0)
print(f"household_df: {len(household_df)} rows, one per household")
print(f"Positive rate: {household_df['HAS_HYBRID'].mean():.4f}")
household_df.head()

household_df: 7417 rows, one per household
Positive rate: 0.0875


## Step 2: baseline, demographics only (matches app/streamlit_app.py)

In [ ]:
demo_features = ['HHFAMINC_IMP', 'HHSIZE', 'HHVEHCNT', 'URBRUR_BIN', 'URBANSIZE',
                  'DRVRCNT', 'HOMEOWN', 'LIF_CYC', 'WRKCOUNT', 'MSASIZE', 'CNTTDHH']

X = household_df[demo_features]
y = household_df['HAS_HYBRID']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)),
])
baseline_model.fit(X_train, y_train)
proba = baseline_model.predict_proba(X_test)[:, 1]
pred = baseline_model.predict(X_test)
print(f"Demographics-only model: ROC-AUC = {roc_auc_score(y_test, proba):.4f}, accuracy = {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=['No hybrid', 'Has hybrid']))

Demographics-only model: ROC-AUC = 0.6792, accuracy = 0.6276
              precision    recall  f1-score   support

   No hybrid       0.95      0.63      0.75      2031
  Has hybrid       0.14      0.63      0.23       195

    accuracy                           0.63      2226
   macro avg       0.54      0.63      0.49      2226
weighted avg       0.88      0.63      0.71      2226



## Step 3: add vehicle-fleet features, without touching `VEHFUEL`/`HYBRID`/`MAKE`

Multi-vehicle households have real, legitimate vehicle-fleet information beyond just a count
(`HHVEHCNT`, already included above): how old the vehicles are, how much they're driven, how
varied the fleet is. None of that requires knowing which vehicle is the hybrid.

In [ ]:
fleet_features = (
    df.groupby('HOUSEID')
    .agg(
        max_vehicle_age=('VEHAGE', 'max'),
        min_vehicle_age=('VEHAGE', 'min'),
        avg_annual_miles=('ANNMILES', 'mean'),
        n_distinct_vehicle_types=('VEHTYPE', 'nunique'),
        pct_commercial=('VEHCOMMERCIAL', lambda s: (s == 1).mean()),
    )
    .reset_index()
)
fleet_features.head()

In [ ]:
household_plus_fleet = household_df.merge(fleet_features, on='HOUSEID', how='left')
expanded_features = demo_features + ['max_vehicle_age', 'min_vehicle_age', 'avg_annual_miles',
                                       'n_distinct_vehicle_types', 'pct_commercial']

X2 = household_plus_fleet[expanded_features]
y2 = household_plus_fleet['HAS_HYBRID']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)

fleet_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)),
])
fleet_model.fit(X2_train, y2_train)
proba2 = fleet_model.predict_proba(X2_test)[:, 1]
pred2 = fleet_model.predict(X2_test)
print(f"Demographics + fleet-age/mileage model: ROC-AUC = {roc_auc_score(y2_test, proba2):.4f}, accuracy = {accuracy_score(y2_test, pred2):.4f}")
print(classification_report(y2_test, pred2, target_names=['No hybrid', 'Has hybrid']))

print(f"Comparison:")
print(f"  Demographics only:             ROC-AUC {roc_auc_score(y_test, proba):.4f}")
print(f"  Demographics + fleet features:  ROC-AUC {roc_auc_score(y2_test, proba2):.4f}")

Demographics + fleet-age/mileage model: ROC-AUC = 0.7131, accuracy = 0.6096
              precision    recall  f1-score   support

   No hybrid       0.96      0.60      0.74      2031
  Has hybrid       0.15      0.72      0.25       195

    accuracy                           0.61      2226
   macro avg       0.55      0.66      0.49      2226
weighted avg       0.89      0.61      0.69      2226

Comparison:
  Demographics only:             ROC-AUC 0.6792
  Demographics + fleet features:  ROC-AUC 0.7131


In [ ]:
coefs = pd.Series(fleet_model.named_steps['lr'].coef_[0], index=expanded_features).sort_values(key=abs, ascending=False)
print("Logistic regression coefficients (standardized), sorted by magnitude:")
print(coefs)

Logistic regression coefficients (standardized), sorted by magnitude:
min_vehicle_age            -0.639496
HHFAMINC_IMP                0.352030
n_distinct_vehicle_types   -0.179008
HHSIZE                     -0.156572
CNTTDHH                     0.124693
URBANSIZE                  -0.114831
avg_annual_miles           -0.112661
DRVRCNT                     0.086336
WRKCOUNT                    0.085898
pct_commercial              0.066744
HHVEHCNT                    0.063000
MSASIZE                     0.055139
LIF_CYC                     0.049979
max_vehicle_age             0.045257
URBRUR_BIN                  0.037687
HOMEOWN                    -0.024068
dtype: float64


## Step 4: a couple more legitimate household variables, and cross-validation instead of one split

Two things worth fixing before trying fancier models. First, the single 70/30 split above has
only ~195 positive test examples, small enough that the AUC could shift meaningfully depending on
which households happen to land in the test set. Cross-validation gives a steadier read. Second,
there are a handful of other genuinely household/person-level NHTS columns not yet tried:
`NUMADLT` (adults in household), `HOMETYPE`, `RAIL` (rail transit access, recoded to a 0/1 flag),
`CDIVMSAR`/`CENSUS_R` (census division/region), `PPT517` (kids 5-17), `YOUNGCHILD`, `RESP_CNT`.
None of these are vehicle-specific or protected demographic attributes (race/ethnicity are
deliberately left out here, they're in the data but not appropriate to base a predictive model on
for a public demo).

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

extra_cols = ['NUMADLT', 'HOMETYPE', 'RAIL', 'CDIVMSAR', 'PPT517', 'YOUNGCHILD', 'RESP_CNT', 'CENSUS_R']
household_df2 = df[['HOUSEID'] + extra_cols].drop_duplicates(subset='HOUSEID').reset_index(drop=True)
household_df2['RAIL_ACCESS'] = household_df2['RAIL'].apply(lambda x: 1 if x == 1 else 0)

full = household_plus_fleet.merge(household_df2.drop(columns='RAIL'), on='HOUSEID', how='left')
extra_features = ['NUMADLT', 'HOMETYPE', 'RAIL_ACCESS', 'CDIVMSAR', 'PPT517', 'YOUNGCHILD', 'RESP_CNT', 'CENSUS_R']
all_features = expanded_features + extra_features

X_all = full[all_features]
y_all = full['HAS_HYBRID']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(model, X, y, name):
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    print(f"{name}: CV ROC-AUC = {scores.mean():.4f} +/- {scores.std():.4f}")
    return scores.mean()

lr_base_cv = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])
cv_auc(lr_base_cv, full[expanded_features], y_all, "LogReg, demographics + fleet (5-fold CV)")

lr_all_cv = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])
cv_auc(lr_all_cv, X_all, y_all, "LogReg, + extra household variables (5-fold CV)")

rf_all = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
cv_auc(rf_all, X_all, y_all, "RandomForest, + extra household variables (5-fold CV)")

pos, neg = (y_all == 1).sum(), (y_all == 0).sum()
xgb_clf = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
                             colsample_bytree=0.8, random_state=42, scale_pos_weight=neg / pos, eval_metric='auc')
cv_auc(xgb_clf, X_all, y_all, "XGBoost, + extra household variables (5-fold CV)")

LogReg, demographics + fleet (5-fold CV): CV ROC-AUC = 0.7035 +/- 0.0268
LogReg, + extra household variables (5-fold CV): CV ROC-AUC = 0.7126 +/- 0.0167
RandomForest, + extra household variables (5-fold CV): CV ROC-AUC = 0.7025 +/- 0.0203
XGBoost, + extra household variables (5-fold CV): CV ROC-AUC = 0.7165 +/- 0.0211


Cross-validation confirms the single-split numbers weren't a fluke (0.70-0.71 range), and the
extra household variables genuinely help, `n=712` LogReg went from 0.70 to 0.71, and XGBoost with
those variables is the best so far at 0.7165. Also notice the fold-to-fold spread (roughly
±0.02-0.03 AUC) mentioned above, that's the real uncertainty band on all of these numbers given
only ~1,400 total positive households.

## Step 5: tune XGBoost properly instead of guessing hyperparameters

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [150, 300, 500],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.02, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}
base_xgb = xgb.XGBClassifier(random_state=42, scale_pos_weight=neg / pos, eval_metric='auc')
search = RandomizedSearchCV(base_xgb, param_dist, n_iter=25, scoring='roc_auc', cv=skf, random_state=42, n_jobs=-1)
search.fit(X_all, y_all)

print("Best CV ROC-AUC:", search.best_score_)
print("Best params:", search.best_params_)

best_xgb = search.best_estimator_
importances = pd.Series(best_xgb.feature_importances_, index=all_features).sort_values(ascending=False)
print("\nTop feature importances:")
print(importances.head(10))

Best CV ROC-AUC: 0.7323752871782994
Best params: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.02, 'colsample_bytree': 0.6}

Top feature importances:
min_vehicle_age    0.100463
HHFAMINC_IMP       0.086105
CENSUS_R           0.079813
CDIVMSAR           0.053347
HHVEHCNT           0.049441
WRKCOUNT           0.044674
max_vehicle_age    0.040616
HOMETYPE           0.040138
RESP_CNT           0.038054
URBANSIZE          0.037784
dtype: float32


## Step 6: person-level features and a vehicle price-tier proxy

Two of the ideas from the first pass, tried for real:

**Commute distance, age, and education from `perpub.csv`** (not touched until now). It's a
person-level file, multiple rows per household, so `GCDWORK` (commute distance), `R_AGE`, and
`EDUC` get aggregated to one row per household (average age, highest education level in the
household, average commute distance among workers, capped at 100 miles to blunt a few extreme
outliers).

**A vehicle price-tier proxy.** NHTS doesn't include MSRP or purchase price anywhere in the
public files, the single most obvious missing predictor per `reference/CHALLENGES.md`. As a
rough stand-in, `MAKE` (vehicle brand) is genuinely in the data, so households can be flagged by
what fraction of their fleet is a traditionally premium brand (Lincoln, Cadillac, Audi, BMW,
Mercedes-Benz, Volvo, Acura, Infiniti, Lexus). **This is a coarse, subjective proxy I built by
hand from general brand-positioning knowledge, not real MSRP data**, so treat it as a rough
signal, not ground truth. It doesn't touch `VEHFUEL` or `HYBRID`, so it isn't leakage, just an
imperfect substitute for the real missing variable.

In [ ]:
person_data = pd.read_csv('../data/perpub.csv')

person_agg = person_data.groupby('HOUSEID').agg(
    avg_age=('R_AGE', 'mean'),
    max_educ=('EDUC', lambda s: s[s > 0].max() if (s > 0).any() else 0),
    avg_commute_dist=('GCDWORK', lambda s: s[s >= 0].mean() if (s >= 0).any() else 0),
).reset_index()
person_agg['avg_commute_dist'] = person_agg['avg_commute_dist'].clip(upper=100)

# vehicle brand price-tier proxy, hand-built brand tiering (see markdown above), not real MSRP data
PREMIUM_MAKES = {13, 19, 32, 34, 42, 51, 54, 58, 59}  # Lincoln, Cadillac, Audi, BMW, Mercedes, Volvo, Acura, Infiniti, Lexus
df['is_premium_make'] = df['MAKE'].isin(PREMIUM_MAKES).astype(int)
brand_features = df.groupby('HOUSEID').agg(pct_premium_brand=('is_premium_make', 'mean')).reset_index()

full = full.merge(brand_features, on='HOUSEID', how='left').merge(person_agg, on='HOUSEID', how='left')
new_features = ['pct_premium_brand', 'avg_age', 'max_educ', 'avg_commute_dist']
all_features = all_features + new_features

X = full[all_features]
y = full['HAS_HYBRID']

tuned_xgb = xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.02, subsample=0.7,
                               colsample_bytree=0.6, min_child_weight=1, random_state=42,
                               scale_pos_weight=neg / pos, eval_metric='auc')
cv_auc(tuned_xgb, full[all_features[:-4]], y_all, "XGBoost (tuned), previous feature set")
cv_auc(tuned_xgb, X, y, "XGBoost (tuned), + brand-tier + person-level features")

XGBoost (tuned), previous feature set: CV ROC-AUC = 0.7324 +/- 0.0219
XGBoost (tuned), + brand-tier + person-level features: CV ROC-AUC = 0.7515 +/- 0.0239


That's the biggest single jump so far, 0.73 to 0.75. Worth being honest about why: some of that
is probably the brand-tier proxy genuinely capturing purchase-price information the model didn't
have before, and some of it is likely `avg_commute_dist`/`max_educ` correlating with income in a
way `HHFAMINC_IMP` alone doesn't fully capture. With 8 more features and the same ~1,400 positive
households, there's also more room for the model to fit noise, the CV fold spread (±0.024) is
about the same as before, so this isn't obviously overfitting, but it's worth staying skeptical of.

## Step 7: does rebalancing the classes actually help?

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# what we've been using all along
cv_auc(tuned_xgb, X, y, "XGBoost + scale_pos_weight (current approach)")

# no rebalancing at all
xgb_noweight = xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.02, subsample=0.7,
                                  colsample_bytree=0.6, min_child_weight=1, random_state=42, eval_metric='auc')
cv_auc(xgb_noweight, X, y, "XGBoost, no rebalancing at all")

# SMOTE oversampling, fit inside each CV fold so synthetic points never leak into the test fold
smote_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.02, subsample=0.7,
                               colsample_bytree=0.6, min_child_weight=1, random_state=42, eval_metric='auc')),
])
cv_auc(smote_xgb, X, y, "XGBoost + SMOTE oversampling (in-fold)")

smote_lr = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(random_state=42, max_iter=1000)),
])
cv_auc(smote_lr, X, y, "LogReg + SMOTE oversampling (in-fold)")

XGBoost + scale_pos_weight (current approach): CV ROC-AUC = 0.7515 +/- 0.0239
XGBoost, no rebalancing at all: CV ROC-AUC = 0.7529 +/- 0.0202
XGBoost + SMOTE oversampling (in-fold): CV ROC-AUC = 0.6850 +/- 0.0236
LogReg + SMOTE oversampling (in-fold): CV ROC-AUC = 0.7319 +/- 0.0167


Not the result I expected going in, but a real one: rebalancing barely moves ROC-AUC either way
for XGBoost here (0.7515 weighted vs 0.7529 with no rebalancing at all, within noise), and SMOTE
actually makes XGBoost noticeably worse (0.685). ROC-AUC measures ranking ability across every
possible threshold, and `scale_pos_weight` mostly shifts *where* the decision boundary sits, not
how well the model ranks positives above negatives, so a boosted tree model that already handles
imbalance reasonably well via its loss function doesn't gain much from it. SMOTE's synthetic
points are linear interpolations between real households in an 8-continuous/19-categorical
feature space, and interpolating between two arbitrary hybrid-owning households doesn't
necessarily produce a realistic household, which likely explains why it hurt rather than helped.

The lesson: rebalancing techniques are not a free win, they're worth testing against a genuine
no-rebalancing baseline rather than assumed. What rebalancing *does* change is where you place the
decision threshold, which is the next question.

## Step 8: the decision threshold matters more than the rebalancing method did

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, accuracy_score

# out-of-fold probabilities: every household gets a prediction from a fold that never trained on it
oof_proba = cross_val_predict(tuned_xgb, X, y, cv=skf, method='predict_proba')[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y, oof_proba)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1s[:-1])
best_threshold = thresholds[best_idx]
print(f"Best-F1 threshold: {best_threshold:.3f}")
print()

for t in [0.10, 0.20, 0.30, 0.50, round(best_threshold, 2)]:
    pred_t = (oof_proba >= t).astype(int)
    print(f"threshold={t:.2f}: precision={precision_score(y, pred_t):.3f}  recall={recall_score(y, pred_t):.3f}  "
          f"f1={f1_score(y, pred_t):.3f}  accuracy={accuracy_score(y, pred_t):.3f}  flagged={pred_t.sum()}/{len(y)}")

Best-F1 threshold: 0.642

threshold=0.10: precision=0.094  recall=0.989  f1=0.172  accuracy=0.165  flagged=6825/7417
threshold=0.20: precision=0.106  recall=0.957  f1=0.191  accuracy=0.291  flagged=5851/7417
threshold=0.30: precision=0.124  recall=0.904  f1=0.218  accuracy=0.431  flagged=4746/7417
threshold=0.50: precision=0.184  recall=0.641  f1=0.285  accuracy=0.719  flagged=2266/7417
threshold=0.64: precision=0.257  recall=0.407  f1=0.315  accuracy=0.845  flagged=1027/7417


ROC-AUC doesn't care where the cutoff is, but every classification metric does, and 0.5 is just a
default, not something earned. At threshold 0.10, the model flags 92% of all households
(6,825 of 7,417) and catches 98.9% of hybrid owners, useless as a filter, but great if the cost of
a false negative is much higher than a false positive (e.g., routing every plausible lead to a
marketing campaign). At threshold 0.64, it flags only 1,027 households, catches 41% of hybrid
owners, but 1 in 4 of its flags is a real hit, a genuinely more useful list if the cost of
contacting the wrong household matters. Which threshold is "right" depends entirely on what this
model would actually be used for, not on a metric.

## Step 9: does stacking multiple models beat the best single model?

Stacking trains several "base" models, then trains a meta-learner on their out-of-fold
predictions to learn how to combine them. Worth trying since XGBoost, Logistic Regression, and
Random Forest don't necessarily make the same mistakes, if their errors are different enough,
combining them can beat any one alone.

In [ ]:
from sklearn.ensemble import StackingClassifier

lr_pipe = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))])
rf_clf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)

print("Individual models, for comparison:")
cv_auc(tuned_xgb, X, y, "XGBoost (tuned) alone")
cv_auc(lr_pipe, X, y, "Logistic Regression alone")
cv_auc(rf_clf, X, y, "Random Forest alone")

stack = StackingClassifier(
    estimators=[('xgb', tuned_xgb), ('lr', lr_pipe), ('rf', rf_clf)],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, stack_method='predict_proba',
)
cv_auc(stack, X, y, "Stacked (XGBoost + LogReg + RF -> LogReg meta-learner)")

# a cheaper alternative to a trained meta-learner: just average the three models' probabilities
xgb_oof = cross_val_predict(tuned_xgb, X, y, cv=skf, method='predict_proba')[:, 1]
lr_oof = cross_val_predict(lr_pipe, X, y, cv=skf, method='predict_proba')[:, 1]
rf_oof = cross_val_predict(rf_clf, X, y, cv=skf, method='predict_proba')[:, 1]
avg_proba = (xgb_oof + lr_oof + rf_oof) / 3
print(f"Simple average of the three models' out-of-fold probabilities: ROC-AUC = {roc_auc_score(y, avg_proba):.4f}")

Individual models, for comparison:
XGBoost (tuned) alone: CV ROC-AUC = 0.7515 +/- 0.0239
Logistic Regression alone: CV ROC-AUC = 0.7360 +/- 0.0158
Random Forest alone: CV ROC-AUC = 0.7206 +/- 0.0263
Stacked (XGBoost + LogReg + RF -> LogReg meta-learner): CV ROC-AUC = 0.7505 +/- 0.0198
Simple average of the three models' out-of-fold probabilities: ROC-AUC = 0.7481


Stacking (0.7505) and simple averaging (0.7481) both land at or slightly below XGBoost alone
(0.7515), not a meaningful win either way. That's a real, useful negative result: XGBoost isn't
leaving much on the table that Logistic Regression or Random Forest can clean up, their errors
overlap too much to gain from combining, and Random Forest specifically (0.72) is dragging the
blend down more than the other two are lifting it. Stacking helps most when the base models are
meaningfully diverse; three models trained on the same 27 features rarely are. Not worth the
added complexity here.

## Step 10: XGBoost's probabilities look confident, are they actually right?

`scale_pos_weight` (used throughout to handle the class imbalance) helps XGBoost *rank* hybrid
owners above non-owners, which is what ROC-AUC measures, but it does this by inflating the raw
scores to compensate for how rare the positive class is. That can leave the actual probability
numbers badly wrong even while the ranking is fine. Checking this directly: bucket households by
predicted probability, and see what fraction in each bucket are actually hybrid owners.

In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

print(f"Brier score, lower is better, 0 = perfect: {brier_score_loss(y, xgb_oof):.4f}")
frac_pos, mean_pred = calibration_curve(y, xgb_oof, n_bins=10, strategy='quantile')
print("\npredicted probability  ->  actual observed frequency in that bucket")
for p, f in zip(mean_pred, frac_pos):
    print(f"  {p:.3f}  ->  {f:.3f}")

Brier score, lower is better, 0 = perfect: 0.1840

predicted probability  ->  actual observed frequency in that bucket
  0.065  ->  0.012
  0.158  ->  0.022
  0.226  ->  0.030
  0.293  ->  0.053
  0.357  ->  0.057
  0.416  ->  0.065
  0.474  ->  0.086
  0.539  ->  0.107
  0.627  ->  0.168
  0.753  ->  0.276


Badly miscalibrated. Households the model scores at "75% probability" are actually hybrid owners
only 27.6% of the time, and the model is overconfident at every bucket, "6.5%" really means 1.2%,
"53.9%" really means 10.7%. The ranking is still useful (that's what the 0.75 AUC captures, higher
scores really do mean more likely), but if this model were used anywhere the raw number gets
read literally ("this household has a 75% chance"), it would be actively misleading.

In [ ]:
calibrated = CalibratedClassifierCV(tuned_xgb, method='sigmoid', cv=skf)
cal_oof = cross_val_predict(calibrated, X, y, cv=skf, method='predict_proba')[:, 1]

print(f"Brier score after sigmoid (Platt) calibration: {brier_score_loss(y, cal_oof):.4f}  (was {brier_score_loss(y, xgb_oof):.4f})")
print(f"ROC-AUC after calibration: {roc_auc_score(y, cal_oof):.4f}  (was {roc_auc_score(y, xgb_oof):.4f}, should be ~unchanged)")

frac_pos2, mean_pred2 = calibration_curve(y, cal_oof, n_bins=10, strategy='quantile')
print("\npredicted probability  ->  actual observed frequency in that bucket, after calibration")
for p, f in zip(mean_pred2, frac_pos2):
    print(f"  {p:.3f}  ->  {f:.3f}")

Brier score after sigmoid (Platt) calibration: 0.0735  (was 0.1840)
ROC-AUC after calibration: 0.7499  (was 0.7498, should be ~unchanged)

predicted probability  ->  actual observed frequency in that bucket, after calibration
  0.016  ->  0.009
  0.023  ->  0.027
  0.031  ->  0.031
  0.041  ->  0.040
  0.055  ->  0.066
  0.071  ->  0.067
  0.091  ->  0.080
  0.118  ->  0.113
  0.166  ->  0.168
  0.263  ->  0.272


That's the calibration story working exactly as advertised: Brier score more than halves
(0.184 to 0.074), the bucketed predictions now line up closely with reality (predicted 0.263,
actually 0.272), and ROC-AUC barely moves (0.7498 to 0.7499), confirming calibration only fixes
what the numbers *mean*, not how well the model *ranks*. Any version of this model that reports a
probability to an end user, rather than just a yes/no flag or a ranked list, should go through
this step. The Streamlit app currently shows the raw, uncalibrated probability, worth fixing.

## Step 11: trip-level data and respondent sex, two more real tests

ADSC shared `trippub.csv` (NHTS trip-level file) and pointed at a real academic finding worth
testing: Williams (2023, *Transport Policy*) found that once EV adopters are compared against
typical new-car buyers rather than the general population, income looks far less exceptional
than assumed, the two things that actually distinguish EV consumers are home ownership and
sex/gender. We already have `HOMEOWN`; `R_SEX_IMP` from `perpub.csv` is new.

Two caveats going in: `trippub.csv` only covers 6,188 of our 7,417 households (83.4%), so any
trip-derived feature needs a missing-data strategy (imputed here with the median plus a
`has_trip_data` flag). And none of the trip fields touch `VEHFUEL`/`HYBRID`/`MAKE`, so this isn't
a leakage risk, just an open question of whether it adds real signal.

In [ ]:
trip_data = pd.read_csv('../data/trippub.csv')

trip_clean = trip_data.copy()
trip_clean['TRPMILES'] = trip_clean['TRPMILES'].where(trip_clean['TRPMILES'] >= 0)
trip_clean['GASPRICE'] = trip_clean['GASPRICE'].where(trip_clean['GASPRICE'] > 0) / 100  # cents -> dollars

trip_agg = trip_data.groupby('HOUSEID').agg(n_trips=('TRIPID', 'count')).reset_index()
trip_agg2 = trip_clean.groupby('HOUSEID').agg(
    total_vmt=('TRPMILES', 'sum'), avg_trip_miles=('TRPMILES', 'mean'),
    avg_gas_price=('GASPRICE', 'mean'), pct_transit_trips=('PUBTRANS', lambda s: (s == 1).mean()),
).reset_index()
trip_agg = trip_agg.merge(trip_agg2, on='HOUSEID', how='left')
print(f"Trip data covers {trip_agg['HOUSEID'].nunique()} of {full['HOUSEID'].nunique()} households "
      f"({trip_agg['HOUSEID'].nunique() / full['HOUSEID'].nunique() * 100:.1f}%)")

sex_agg = person_data.groupby('HOUSEID').agg(pct_male=('R_SEX_IMP', lambda s: (s == 1).mean())).reset_index()

full2 = full.merge(trip_agg, on='HOUSEID', how='left').merge(sex_agg, on='HOUSEID', how='left')
full2['has_trip_data'] = full2['n_trips'].notna().astype(int)
for c in ['n_trips', 'total_vmt', 'avg_trip_miles', 'avg_gas_price', 'pct_transit_trips']:
    full2[c] = full2[c].fillna(full2[c].median())

trip_features = ['n_trips', 'total_vmt', 'avg_trip_miles', 'avg_gas_price', 'pct_transit_trips', 'has_trip_data']
y2 = full2['HAS_HYBRID']

cv_auc(tuned_xgb, full2[all_features], y2, "Baseline (Step 6-8 feature set, no sex/trip)")
cv_auc(tuned_xgb, full2[all_features + ['pct_male']], y2, "+ pct_male (R_SEX)")
cv_auc(tuned_xgb, full2[all_features + trip_features], y2, "+ trip aggregates")
cv_auc(tuned_xgb, full2[all_features + ['pct_male'] + trip_features], y2, "+ both")

# isolate the missing-data confound: same comparison, only households that actually have trip data
sub = full2[full2['has_trip_data'] == 1]
cv_auc(tuned_xgb, sub[all_features], sub['HAS_HYBRID'], f"Baseline, trip-data-only subset (n={len(sub)})")
cv_auc(tuned_xgb, sub[all_features + trip_features[:-1]], sub['HAS_HYBRID'], f"+ trip aggregates, same subset (n={len(sub)})")

Trip data covers 6188 of 7417 households (83.4%)
Baseline (Step 6-8 feature set, no sex/trip): CV ROC-AUC = 0.7515 +/- 0.0239
+ pct_male (R_SEX): CV ROC-AUC = 0.7484 +/- 0.0236
+ trip aggregates: CV ROC-AUC = 0.7461 +/- 0.0219
+ both: CV ROC-AUC = 0.7463 +/- 0.0225
Baseline, trip-data-only subset (n=5937): CV ROC-AUC = 0.7442 +/- 0.0253
+ trip aggregates, same subset (n=5937): CV ROC-AUC = 0.7423 +/- 0.0252


Another honest negative. `pct_male` alone costs a little (0.7515 to 0.7484), trip aggregates cost
a little more (0.7461), together they don't recover it (0.7463). Restricting to only the 5,937
households that actually have trip data, removing the imputation-noise question entirely, the
gap holds (0.7442 baseline vs. 0.7423 with trip features). All of these differences sit inside
the ±0.02-0.024 fold-to-fold noise band, so nothing here is a disaster, but nothing earns a place
in the feature set either.

Worth being honest about why the Williams (2023) finding didn't transfer: that study compared EV
*rebate recipients* specifically to *new-car buyers* specifically, both narrower and different
populations than "NHTS households that happen to own a hybrid" vs. "households that don't." Sex
distinguishing EV adopters from the pool of people who just bought a new car is a different claim
than sex predicting hybrid ownership across an entire national household sample that includes
households who haven't bought a car in a decade. The literature finding is real, it just doesn't
directly transfer to this exact prediction task.

## Step 12: real vehicle price data, a dead end worth documenting

Checked three paths for real MSRP/price data to replace the hand-built brand-tier proxy from
Step 6:

1. `ev-database.org` (flagged in `reference/Compendium Relevant Studies.docx`), a European EV
   database. Even setting aside access issues, it's EUR pricing on EU-market trims, a real
   mismatch for US NHTS households.
2. Free US vehicle datasets (`us-car-models-data`, back4app's car database), these have
   make/model/year/trim but no price field at all.
3. Kelley Blue Book / Cox Automotive publishes real 2022 US average-transaction-price data by
   brand, but the actual per-brand numbers live in chart images inside their articles, not as
   extractable text or a downloadable table. Reading numbers off a chart image isn't reliable
   enough to treat as real data.

A genuine structured make/model/year to MSRP dataset exists, but only as a paid commercial
product. Real price data stays a documented gap rather than a feature, the brand-tier proxy from
Step 6 (hand-built, clearly labeled as an estimate) is what's actually in the model.

## Takeaways (final)

| Model | Features | CV ROC-AUC |
|---|---|---|
| Logistic Regression | demographics only | 0.68 |
| Logistic Regression | + vehicle-fleet age/mileage | 0.70 |
| Logistic Regression | + extra household variables | 0.71 |
| XGBoost (tuned) | same expanded feature set | 0.73 |
| **XGBoost (tuned)** | **+ brand-tier proxy + person-level features** | **0.75** |
| XGBoost (tuned) | + trip aggregates and/or respondent sex | 0.75 (no improvement) |
| Stacked / averaged ensemble | same features | 0.75 (no improvement) |

**0.75 ROC-AUC, calibrated, is the honest ceiling** reached on this NHTS sample with the data
available. What moved the number: fixing the household-level dedup and dropping the leaky
vehicle-identity columns (0.68 baseline), fleet-level vehicle features, extra household
demographics, hyperparameter tuning, and the brand-tier proxy (0.68 to 0.75 combined). What
didn't: SMOTE, stacking, respondent sex, and trip-level aggregates, each tested honestly rather
than assumed, and each a real (if negative) finding in its own right.

What's still genuinely missing: real vehicle price/MSRP data (checked three sources, no viable
free path found, see Step 12), charging infrastructure and incentive data (not in NHTS at all).
Those are the two levers most likely to move this further, and both require data this project
doesn't currently have access to.